# ASAP-AES Baseline Notebook

This notebook presents the current paper-style AES baseline in a review-friendly format.

Pipeline:

1. Load ASAP-AES
2. Create reproducible internal train/val/test splits per `essay_set`
3. Generate train-only weak labels with signal clustering
4. Extract interpretable features
5. Train one positive linear regression model per `essay_set`
6. Evaluate with QWK, MAE, and Pearson
7. Save outputs under `results/`

The notebook reuses the current `src/` implementation and teammate helper logic from `src/matrix.py` and `src/signal_pred.py`.

## 1. Imports and paths

In [1]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

CANDIDATES = [Path.cwd(), Path.cwd().parent]
PROJECT_ROOT = next(
    (path for path in CANDIDATES if (path / "src").exists() and (path / "README.md").exists()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError("Could not locate the repository root from this notebook.")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.aes_features import extract_feature_splits, write_feature_splits
from src.data_loading import DEFAULT_DATA_DIR, load_asap_split
from src.run_aes_baseline import (
    DEFAULT_METRIC_DIR,
    DEFAULT_MODEL_DIR,
    DEFAULT_PREDICTION_DIR,
    DEFAULT_PROCESSED_DIR,
    DEFAULT_SEED,
    DEFAULT_TEST_SIZE,
    DEFAULT_VAL_SIZE,
    DEFAULT_WEAK_LABEL_DIR,
    build_aes_splits,
    generate_weak_labels,
    train_predict_evaluate,
    write_metric_outputs,
    write_model_outputs,
    write_prediction_outputs,
    write_processed_splits,
    write_weak_label_outputs,
)

DATA_DIR = DEFAULT_DATA_DIR
PROCESSED_DIR = DEFAULT_PROCESSED_DIR
WEAK_LABEL_DIR = DEFAULT_WEAK_LABEL_DIR
FEATURE_DIR = PROJECT_ROOT / "results" / "features" / "asap-aes"
PREDICTION_DIR = DEFAULT_PREDICTION_DIR
METRIC_DIR = DEFAULT_METRIC_DIR
MODEL_DIR = DEFAULT_MODEL_DIR

SEED = DEFAULT_SEED
VAL_SIZE = DEFAULT_VAL_SIZE
TEST_SIZE = DEFAULT_TEST_SIZE
MIN_DF = 5
MAX_ITER = 100
EPS = 1e-5

print(f"PROJECT_ROOT: {PROJECT_ROOT}")
print(f"DATA_DIR: {DATA_DIR}")

PROJECT_ROOT: /home/harsh/Desktop/Socks
DATA_DIR: /home/harsh/Desktop/Socks/data/asap-aes


## 2. Load ASAP-AES data

In [2]:
raw_train = load_asap_split("train", data_dir=DATA_DIR)
print(raw_train.shape)
display(raw_train[["essay_id", "essay_set", "essay", "domain1_score"]].head())
display(raw_train["essay_set"].value_counts().sort_index().rename("rows_per_essay_set"))

(12976, 28)


,essay_id,essay_set,essay,domain1_score
0,1,1,"Dear local newspaper, I think effects computer...",8
1,2,1,"Dear @CAPS1 @CAPS2, I believe that using compu...",9
2,3,1,"Dear, @CAPS1 @CAPS2 @CAPS3 More and more peopl...",7
3,4,1,"Dear Local Newspaper, @CAPS1 I have found that...",10
4,5,1,"Dear @LOCATION1, I know having computers has a...",8


1    1783
2    1800
3    1726
4    1770
5    1805
6    1800
7    1569
8     723
Name: rows_per_essay_set, dtype: int64

## 3. Create reproducible internal train/val/test splits per `essay_set`

In [3]:
processed_splits, split_summary = build_aes_splits(
    data_dir=DATA_DIR,
    val_size=VAL_SIZE,
    test_size=TEST_SIZE,
    seed=SEED,
)
processed_paths = write_processed_splits(processed_splits, split_summary, output_dir=PROCESSED_DIR)

display(split_summary)
processed_paths

,essay_set,split,n,score_min,score_max
0,1,test,357,2.0,12.0
1,1,train,1069,2.0,12.0
2,1,val,357,2.0,12.0
3,2,test,360,1.0,6.0
4,2,train,1080,1.0,6.0
5,2,val,360,1.0,6.0
6,3,test,346,0.0,3.0
7,3,train,1035,0.0,3.0
8,3,val,345,0.0,3.0
9,4,test,354,0.0,3.0


{'train': PosixPath('/home/harsh/Desktop/Socks/results/processed/asap-aes/train.csv'),
 'val': PosixPath('/home/harsh/Desktop/Socks/results/processed/asap-aes/val.csv'),
 'test': PosixPath('/home/harsh/Desktop/Socks/results/processed/asap-aes/test.csv'),
 'split_summary': PosixPath('/home/harsh/Desktop/Socks/results/processed/asap-aes/split_summary.csv')}

## 4. Generate train-only weak labels using signal clustering

In [4]:
weak_labels, weak_diagnostics = generate_weak_labels(
    processed_splits["train"],
    min_df=MIN_DF,
    max_iter=MAX_ITER,
    eps=EPS,
)
weak_paths = write_weak_label_outputs(weak_labels, weak_diagnostics, WEAK_LABEL_DIR)

display(weak_diagnostics)
display(weak_labels.head())
weak_paths

,essay_set,n,vocabulary_size,init_length_mean,weak_mean,weak_std,length_weak_pearson
0,1,1069,2664,2021.875585,0.607546,0.185427,0.864779
1,2,1080,2515,2065.941667,0.568646,0.194763,0.868554
2,3,1035,1235,585.407729,0.584282,0.192513,0.830955
3,4,1062,965,504.994350,0.552716,0.208084,0.739281
4,5,1083,1061,677.563250,0.557060,0.213036,0.826373
5,6,1080,1270,889.265741,0.463404,0.214849,0.572416
6,7,941,1469,837.127524,0.552451,0.193435,0.870257
7,8,433,2432,3159.642032,0.621320,0.216757,0.744219


,essay_id,essay_set,weak_label_raw,weak_label_normalized
0,1,1,-0.082280,0.592304
1,2,1,0.934825,0.780726
2,3,1,0.000461,0.607632
3,4,1,1.440387,0.874383
4,5,1,0.774943,0.751107


{'train': PosixPath('/home/harsh/Desktop/Socks/results/weak_labels/asap-aes/train_weak_labels.csv'),
 'diagnostics': PosixPath('/home/harsh/Desktop/Socks/results/weak_labels/asap-aes/train_weak_label_diagnostics.csv')}

## 5. Extract interpretable features

In [5]:
feature_splits = extract_feature_splits(processed_splits)
feature_paths = write_feature_splits(feature_splits, output_dir=FEATURE_DIR)

display(feature_splits["train"].head())
feature_paths

,essay_id,essay_set,split,word_count,character_count,sentence_count,average_word_length,unique_word_count,type_token_ratio,long_word_count,punctuation_count,digit_count,paragraph_count
0,1,1,train,350,1875,16,4.237143,166,0.474286,59,55,5,1
1,2,1,train,423,2288,20,4.312057,196,0.463357,80,46,10,1
2,3,1,train,283,1541,14,4.342756,151,0.533569,53,34,7,1
3,4,1,train,530,3165,27,4.813208,251,0.473585,131,91,41,1
4,5,1,train,473,2569,30,4.334038,201,0.424947,87,55,4,1


{'train': PosixPath('/home/harsh/Desktop/Socks/results/features/asap-aes/train_features.csv'),
 'val': PosixPath('/home/harsh/Desktop/Socks/results/features/asap-aes/val_features.csv'),
 'test': PosixPath('/home/harsh/Desktop/Socks/results/features/asap-aes/test_features.csv')}

## 6. Train positive linear regression per `essay_set`

In [6]:
predictions, metrics, coefficients = train_predict_evaluate(
    processed_splits=processed_splits,
    feature_splits=feature_splits,
    weak_labels=weak_labels,
)

display(coefficients.head(20))

,essay_set,feature,coefficient,scaler_mean,scaler_scale,intercept
0,1,average_word_length,0.009149,4.332942,0.253591,0.607546
1,1,character_count,0.000000,2021.875585,696.535138,0.607546
2,1,digit_count,0.004762,9.711880,10.965003,0.607546
3,1,long_word_count,0.000000,63.990645,27.614781,0.607546
4,1,paragraph_count,0.000000,1.000000,1.000000,0.607546
5,1,punctuation_count,0.004145,53.218896,26.301461,0.607546
6,1,sentence_count,0.014418,23.058934,9.399226,0.607546
7,1,type_token_ratio,0.015314,0.470628,0.073585,0.607546
8,1,unique_word_count,0.141715,168.777362,50.380229,0.607546
9,1,word_count,0.016769,369.463050,122.822656,0.607546


## 7. Evaluate with QWK, MAE, and Pearson

In [7]:
display(metrics)

,split,essay_set,n,qwk,mae,pearson
0,test,1,357,0.726727,0.892956,0.792774
1,test,2,360,0.553829,0.610671,0.680366
2,test,3,346,0.550043,0.487067,0.661328
3,test,4,354,0.512334,0.554605,0.748692
4,test,5,361,0.733326,0.468617,0.810454
5,test,6,360,0.366372,1.005384,0.675249
6,test,7,314,0.691627,2.910775,0.743249
7,test,8,145,0.415414,6.061794,0.550571
8,val,1,357,0.796622,0.833967,0.844843
9,val,2,360,0.604125,0.595043,0.717823


## 8. Save outputs

In [8]:
prediction_paths = write_prediction_outputs(predictions, PREDICTION_DIR)
metric_paths = write_metric_outputs(metrics, METRIC_DIR)
model_paths = write_model_outputs(coefficients, MODEL_DIR)

run_summary = {
    "baseline": "aes_baseline_notebook",
    "macro_metrics": metrics[metrics["essay_set"] == "macro"].to_dict(orient="records"),
}
run_summary_path = PROJECT_ROOT / "results" / "processed" / "asap-aes" / "notebook_run_summary.json"
run_summary_path.parent.mkdir(parents=True, exist_ok=True)
with run_summary_path.open("w", encoding="utf-8") as handle:
    json.dump(run_summary, handle, indent=2)

all_paths = {
    **{f"processed_{name}": path for name, path in processed_paths.items()},
    **{f"weak_{name}": path for name, path in weak_paths.items()},
    **{f"features_{name}": path for name, path in feature_paths.items()},
    **prediction_paths,
    **metric_paths,
    **model_paths,
    "run_summary": run_summary_path,
}
all_paths

{'processed_train': PosixPath('/home/harsh/Desktop/Socks/results/processed/asap-aes/train.csv'),
 'processed_val': PosixPath('/home/harsh/Desktop/Socks/results/processed/asap-aes/val.csv'),
 'processed_test': PosixPath('/home/harsh/Desktop/Socks/results/processed/asap-aes/test.csv'),
 'processed_split_summary': PosixPath('/home/harsh/Desktop/Socks/results/processed/asap-aes/split_summary.csv'),
 'weak_train': PosixPath('/home/harsh/Desktop/Socks/results/weak_labels/asap-aes/train_weak_labels.csv'),
 'weak_diagnostics': PosixPath('/home/harsh/Desktop/Socks/results/weak_labels/asap-aes/train_weak_label_diagnostics.csv'),
 'features_train': PosixPath('/home/harsh/Desktop/Socks/results/features/asap-aes/train_features.csv'),
 'features_val': PosixPath('/home/harsh/Desktop/Socks/results/features/asap-aes/val_features.csv'),
 'features_test': PosixPath('/home/harsh/Desktop/Socks/results/features/asap-aes/test_features.csv'),
 'predictions_val_test': PosixPath('/home/harsh/Desktop/Socks/resul

## 9. Display final macro metrics

In [9]:
macro_metrics = metrics[metrics["essay_set"] == "macro"].copy()
display(macro_metrics)

,split,essay_set,n,qwk,mae,pearson
16,test,macro,2597,0.568709,1.623984,0.707835
17,val,macro,2596,0.604631,1.686444,0.727789


## 10. Notes / limitations

- This notebook reuses the current `src/` AES implementation and preserved teammate helper logic.
- It uses internal `train` / `val` / `test` splits from `training_set_rel3.tsv`.
- Weak labels come from signal clustering only.
- The baseline is intentionally white-box and does not include NLLF, BSQ, or LLM weak labels.